# 01 - Preprocesamiento de imágenes: Eliminación de fondo
### Clasificador de Pokémon por tipo (Generación 1)

Este notebook se encarga de limpiar el dataset original eliminando el fondo de las imágenes utilizando la librería `rembg`. Esto ayuda al modelo a enfocarse en las características del Pokémon y no en el entorno.

**Requerimientos:** `pip install rembg onnxruntime` (u `onnxruntime-gpu` si tienes placa de video).


In [1]:
import os
import random
from pathlib import Path
from PIL import Image
from rembg import remove, new_session

# Ajustar el path si se corre desde dev/
if os.path.basename(os.getcwd()) == "dev":
    os.chdir("..")

print(f"Directorio de trabajo: {os.getcwd()}")

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Directorio de trabajo: c:\Facultad\redes neuronales\redes-pokemon


In [2]:
INPUT_DIR = "data/PokemonData"
OUTPUT_DIR = "data/PokemonDataNoBG"

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png")

if not os.path.exists(INPUT_DIR):
    print(f"Error: No se encontró el directorio {INPUT_DIR}")
else:
    all_images = []
    for root, _, files in os.walk(INPUT_DIR):
        for file in files:
            if file.lower().endswith(VALID_EXTENSIONS):
                all_images.append(os.path.join(root, file))
    print(f"Total imágenes encontradas: {len(all_images)}")

Total imágenes encontradas: 6820


In [3]:
# Inicializar sesión de rembg (descarga el modelo si no existe)
session = new_session("u2net")

100%|########################################| 176M/176M [00:00<00:00, 114GB/s]


In [4]:
processed = 0
errors = 0

for root, _, files in os.walk(INPUT_DIR):
    relative_path = os.path.relpath(root, INPUT_DIR)
    output_folder = os.path.join(OUTPUT_DIR, relative_path)
    os.makedirs(output_folder, exist_ok=True)

    for file in files:
        if not file.lower().endswith(VALID_EXTENSIONS):
            continue
        
        input_path = os.path.join(root, file)
        output_name = Path(file).stem + ".png"
        output_path = os.path.join(output_folder, output_name)

        # Saltar si ya existe para evitar reprocesar
        if os.path.exists(output_path):
            processed += 1
            continue

        try:
            image = Image.open(input_path).convert("RGBA")
            result = remove(image, session=session)
            
            # Crear fondo negro
            black_bg = Image.new("RGBA", result.size, (0, 0, 0, 255))
            result = Image.alpha_composite(black_bg, result).convert("RGB")
            
            result.save(output_path)
            processed += 1

            if processed % 100 == 0:
                print(f"{processed} imágenes procesadas...")

        except Exception as e:
            print(f"Error en {input_path}: {e}")
            errors += 1

print(f"\nProceso finalizado.")
print(f"Procesadas: {processed}")
print(f"Errores: {errors}")

100 imágenes procesadas...


KeyboardInterrupt: 